# Banking Fraud & Transaction Analytics

## Project Objective

Financial institutions process millions of transactions every day. The objective of this project is to build a realistic banking transaction dataset and analyze customer behavior, merchant performance, and fraudulent activities using Python, SQL, and Power BI.

### Tools Used

- Python
- Pandas
- NumPy
- SQL
- Power BI

**Author:** Prajwal Phalke

In [1]:
import pandas as pd
import numpy as np
import random

from datetime import datetime, timedelta

# Reproducibility
random.seed(42)
np.random.seed(42)

print("Libraries imported successfully")



Libraries imported successfully


# 1. Customer Master Table

A customer master table represents the permanent information of every bank customer. Each customer has a unique ID, demographic information, account balance, account type, and city.


In [2]:
# Customer Master Dataset

N_CUSTOMERS = 50000

cities = [
    "Berlin","Munich","Hamburg","Cologne","Frankfurt",
    "Stuttgart","Dresden","Leipzig","Bremen","Hannover"
]

account_types = ["Savings", "Current", "Business"]

customers = pd.DataFrame({
    "customer_id": np.arange(100001, 100001 + N_CUSTOMERS),
    "age": np.random.randint(18, 75, N_CUSTOMERS),
    "city": np.random.choice(cities, N_CUSTOMERS),
    "account_type": np.random.choice(
        account_types,
        N_CUSTOMERS,
        p=[0.55, 0.35, 0.10]
    ),
    "account_balance": np.random.randint(500, 100000, N_CUSTOMERS)
})

customers.head()

,customer_id,age,city,account_type,account_balance
0,100001,56,Cologne,Current,48792
1,100002,69,Berlin,Savings,96812
2,100003,46,Stuttgart,Savings,91883
3,100004,32,Cologne,Savings,35033
4,100005,60,Hamburg,Savings,75819


## Data Quality Check

Before generating transactions, we validate the customer master dataset for shape, missing values, duplicates, and summary statistics.

In [4]:
print("Shape :", customers.shape)
print("\nMissing Values")
print(customers.isnull().sum())

print("\nDuplicate Customer IDs :", customers["customer_id"].duplicated().sum())

customers.describe()

Shape : (50000, 5)

Missing Values
customer_id        0
age                0
city               0
account_type       0
account_balance    0
dtype: int64

Duplicate Customer IDs : 0


,customer_id,age,account_balance
count,50000.000000,50000.000000,50000.000000
mean,125000.500000,46.012240,50154.700180
std,14433.901067,16.414894,28728.953562
min,100001.000000,18.000000,501.000000
25%,112500.750000,32.000000,25199.500000
50%,125000.500000,46.000000,50176.500000
75%,137500.250000,60.000000,75037.250000
max,150000.000000,74.000000,99999.000000


# 2. Merchant Dimension

Merchants represent businesses where customers perform transactions. Each merchant belongs to a business category and operates across German cities.

In [5]:
# Merchant Dimension

merchant_categories = {
    "Grocery": ["Lidl", "Aldi", "REWE", "Edeka"],
    "Fuel": ["Shell", "Aral", "Esso"],
    "Travel": ["DB Bahn", "Lufthansa", "FlixBus"],
    "Shopping": ["Amazon", "Zalando", "IKEA", "MediaMarkt"],
    "Restaurant": ["McDonalds", "Burger King", "Subway", "Nordsee"],
    "Healthcare": ["DM Pharmacy", "Rossmann", "Apotheke"],
    "Entertainment": ["Netflix", "Spotify", "Steam", "CinemaxX"]
}

merchant_records = []

merchant_id = 2001

for category, brands in merchant_categories.items():
    for brand in brands:
        merchant_records.append({
            "merchant_id": merchant_id,
            "merchant_name": brand,
            "merchant_category": category
        })
        merchant_id += 1

merchants = pd.DataFrame(merchant_records)

print("Total merchants:", len(merchants))
merchants.head()

Total merchants: 25


,merchant_id,merchant_name,merchant_category
0,2001,Lidl,Grocery
1,2002,Aldi,Grocery
2,2003,REWE,Grocery
3,2004,Edeka,Grocery
4,2005,Shell,Fuel


## Payment Methods

Customers can pay using multiple banking channels, enabling payment behavior analysis.

In [6]:
payment_methods = [
    "Debit Card",
    "Credit Card",
    "Bank Transfer",
    "Mobile Pay"
]

transaction_channels = [
    "POS",
    "Online",
    "ATM"
]

print(payment_methods)
print(transaction_channels)

['Debit Card', 'Credit Card', 'Bank Transfer', 'Mobile Pay']
['POS', 'Online', 'ATM']


# 3. Transaction Generation

This section creates one million realistic banking transactions using customer behavior, merchant categories, timestamps, and payment channels.

In [7]:
# Transaction Rules

amount_ranges = {
    "Grocery": (10, 180),
    "Fuel": (20, 120),
    "Travel": (40, 1200),
    "Shopping": (30, 700),
    "Restaurant": (8, 90),
    "Healthcare": (5, 250),
    "Entertainment": (5, 60)
}

amount_ranges

{'Grocery': (10, 180),
 'Fuel': (20, 120),
 'Travel': (40, 1200),
 'Shopping': (30, 700),
 'Restaurant': (8, 90),
 'Healthcare': (5, 250),
 'Entertainment': (5, 60)}

In [8]:
# Random Timestamp Generator


start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 12, 31)

def random_timestamp():
    seconds = random.randint(
        0,
        int((end_date - start_date).total_seconds())
    )
    return start_date + timedelta(seconds=seconds)

# Test
random_timestamp()

datetime.datetime(2025, 9, 6, 7, 53, 23)

# 4. Transaction Fact Table

The transaction fact table contains one million banking transactions linking customers with merchants. It represents the operational data used for business intelligence and fraud analytics.

In [9]:
# Generate 1 Million Banking Transactions

N_TRANSACTIONS = 1_000_000

# Random customer selection
customer_ids = np.random.choice(
    customers["customer_id"],
    N_TRANSACTIONS
)

# Random merchant selection
merchant_sample = merchants.sample(
    n=N_TRANSACTIONS,
    replace=True
).reset_index(drop=True)

# Generate timestamps across 2025
date_range = pd.date_range(
    start="2025-01-01",
    end="2025-12-31 23:59:59",
    periods=N_TRANSACTIONS
)

timestamps = np.random.choice(date_range, N_TRANSACTIONS)

# Payment behaviour
payment_method = np.random.choice(
    payment_methods,
    N_TRANSACTIONS,
    p=[0.45, 0.25, 0.20, 0.10]
)

channel = np.random.choice(
    transaction_channels,
    N_TRANSACTIONS,
    p=[0.60, 0.30, 0.10]
)

transactions = pd.DataFrame({
    "transaction_id": np.arange(1, N_TRANSACTIONS + 1),
    "customer_id": customer_ids,
    "timestamp": timestamps,
    "merchant_id": merchant_sample["merchant_id"],
    "merchant_name": merchant_sample["merchant_name"],
    "merchant_category": merchant_sample["merchant_category"],
    "payment_method": payment_method,
    "channel": channel
})

transactions.head()

,transaction_id,customer_id,timestamp,merchant_id,merchant_name,merchant_category,payment_method,channel
0,1,103151,2025-01-21 08:03:49.941314941,2025,CinemaxX,Entertainment,Debit Card,POS
1,2,115351,2025-09-10 00:58:42.877800880,2024,Steam,Entertainment,Bank Transfer,POS
2,3,143291,2025-05-08 22:36:07.103642104,2025,CinemaxX,Entertainment,Mobile Pay,Online
3,4,149230,2025-04-15 11:26:50.308572308,2021,Apotheke,Healthcare,Debit Card,POS
4,5,123853,2025-05-05 22:16:45.635536636,2021,Apotheke,Healthcare,Bank Transfer,POS


In [10]:
# Generate Transaction Amounts

def generate_amount(category):
    low, high = amount_ranges[category]
    return round(np.random.uniform(low, high), 2)

transactions["amount"] = transactions["merchant_category"].apply(generate_amount)

transactions.head()

,transaction_id,customer_id,timestamp,merchant_id,merchant_name,merchant_category,payment_method,channel,amount
0,1,103151,2025-01-21 08:03:49.941314941,2025,CinemaxX,Entertainment,Debit Card,POS,41.36
1,2,115351,2025-09-10 00:58:42.877800880,2024,Steam,Entertainment,Bank Transfer,POS,58.65
2,3,143291,2025-05-08 22:36:07.103642104,2025,CinemaxX,Entertainment,Mobile Pay,Online,17.11
3,4,149230,2025-04-15 11:26:50.308572308,2021,Apotheke,Healthcare,Debit Card,POS,154.81
4,5,123853,2025-05-05 22:16:45.635536636,2021,Apotheke,Healthcare,Bank Transfer,POS,208.99


In [11]:
print("Shape:", transactions.shape)

print("\nMissing Values")
print(transactions.isnull().sum())

print("\nDuplicate Transaction IDs:",
      transactions["transaction_id"].duplicated().sum())

transactions.describe()

Shape: (1000000, 9)

Missing Values
transaction_id       0
customer_id          0
timestamp            0
merchant_id          0
merchant_name        0
merchant_category    0
payment_method       0
channel              0
amount               0
dtype: int64

Duplicate Transaction IDs: 0


,transaction_id,customer_id,timestamp,merchant_id,amount
count,1000000.000000,1000000.000000,1000000,1000000.000000,1000000.000000
mean,500000.500000,125022.270521,2025-07-02 10:34:43.701347840,2012.993233,184.616800
min,1.000000,100001.000000,2025-01-01 00:00:00,2001.000000,5.000000
25%,250000.750000,112521.000000,2025-04-02 02:58:39.819464960,2007.000000,41.800000
50%,500000.500000,125042.000000,2025-07-02 11:45:00.723129856,2013.000000,81.780000
75%,750000.250000,137530.000000,2025-10-01 18:06:41.334389248,2019.000000,196.722500
max,1000000.000000,150000.000000,2025-12-31 23:59:59,2025.000000,1199.980000
std,288675.278933,14429.328893,NaN,7.209061,241.532478


# 5. Data Enrichment

The transaction table is enriched by joining customer demographics with every transaction. This creates a denormalized analytical dataset used for exploratory data analysis and Power BI.

In [13]:
# Time Features

transactions["timestamp"] = pd.to_datetime(transactions["timestamp"])

transactions["year"] = transactions["timestamp"].dt.year
transactions["month"] = transactions["timestamp"].dt.month
transactions["month_name"] = transactions["timestamp"].dt.month_name()

transactions["day"] = transactions["timestamp"].dt.day

transactions["weekday"] = transactions["timestamp"].dt.day_name()

transactions["hour"] = transactions["timestamp"].dt.hour

transactions.head()

,transaction_id,customer_id,timestamp,merchant_id,merchant_name,merchant_category,payment_method,channel,amount,age,city,account_type,account_balance,year,month,month_name,day,weekday,hour
0,1,103151,2025-01-21 08:03:49.941314941,2025,CinemaxX,Entertainment,Debit Card,POS,41.36,38,Munich,Savings,52066,2025,1,January,21,Tuesday,8
1,2,115351,2025-09-10 00:58:42.877800880,2024,Steam,Entertainment,Bank Transfer,POS,58.65,28,Bremen,Current,4380,2025,9,September,10,Wednesday,0
2,3,143291,2025-05-08 22:36:07.103642104,2025,CinemaxX,Entertainment,Mobile Pay,Online,17.11,73,Hamburg,Savings,15541,2025,5,May,8,Thursday,22
3,4,149230,2025-04-15 11:26:50.308572308,2021,Apotheke,Healthcare,Debit Card,POS,154.81,41,Bremen,Savings,67322,2025,4,April,15,Tuesday,11
4,5,123853,2025-05-05 22:16:45.635536636,2021,Apotheke,Healthcare,Bank Transfer,POS,208.99,51,Munich,Business,43646,2025,5,May,5,Monday,22


In [12]:
# Merge Customer Information

transactions = transactions.merge(
    customers,
    on="customer_id",
    how="left"
)

print("Shape:", transactions.shape)
transactions.head()

Shape: (1000000, 13)


,transaction_id,customer_id,timestamp,merchant_id,merchant_name,merchant_category,payment_method,channel,amount,age,city,account_type,account_balance
0,1,103151,2025-01-21 08:03:49.941314941,2025,CinemaxX,Entertainment,Debit Card,POS,41.36,38,Munich,Savings,52066
1,2,115351,2025-09-10 00:58:42.877800880,2024,Steam,Entertainment,Bank Transfer,POS,58.65,28,Bremen,Current,4380
2,3,143291,2025-05-08 22:36:07.103642104,2025,CinemaxX,Entertainment,Mobile Pay,Online,17.11,73,Hamburg,Savings,15541
3,4,149230,2025-04-15 11:26:50.308572308,2021,Apotheke,Healthcare,Debit Card,POS,154.81,41,Bremen,Savings,67322
4,5,123853,2025-05-05 22:16:45.635536636,2021,Apotheke,Healthcare,Bank Transfer,POS,208.99,51,Munich,Business,43646


## Calendar Features

Extract temporal features from timestamps for seasonality, weekday, and hourly transaction analysis.

In [14]:
# Time Features

transactions["timestamp"] = pd.to_datetime(transactions["timestamp"])

transactions["year"] = transactions["timestamp"].dt.year
transactions["month"] = transactions["timestamp"].dt.month
transactions["month_name"] = transactions["timestamp"].dt.month_name()

transactions["day"] = transactions["timestamp"].dt.day

transactions["weekday"] = transactions["timestamp"].dt.day_name()

transactions["hour"] = transactions["timestamp"].dt.hour

transactions.head()

,transaction_id,customer_id,timestamp,merchant_id,merchant_name,merchant_category,payment_method,channel,amount,age,city,account_type,account_balance,year,month,month_name,day,weekday,hour
0,1,103151,2025-01-21 08:03:49.941314941,2025,CinemaxX,Entertainment,Debit Card,POS,41.36,38,Munich,Savings,52066,2025,1,January,21,Tuesday,8
1,2,115351,2025-09-10 00:58:42.877800880,2024,Steam,Entertainment,Bank Transfer,POS,58.65,28,Bremen,Current,4380,2025,9,September,10,Wednesday,0
2,3,143291,2025-05-08 22:36:07.103642104,2025,CinemaxX,Entertainment,Mobile Pay,Online,17.11,73,Hamburg,Savings,15541,2025,5,May,8,Thursday,22
3,4,149230,2025-04-15 11:26:50.308572308,2021,Apotheke,Healthcare,Debit Card,POS,154.81,41,Bremen,Savings,67322,2025,4,April,15,Tuesday,11
4,5,123853,2025-05-05 22:16:45.635536636,2021,Apotheke,Healthcare,Bank Transfer,POS,208.99,51,Munich,Business,43646,2025,5,May,5,Monday,22


# 6. Rule-Based Fraud Detection

Instead of assigning fraud randomly, transactions receive a risk score based on behavioral patterns such as unusual hours, high-value purchases, payment channel, and merchant category.

In [15]:
# Risk Score

transactions["risk_score"] = 0

# Rule 1 — Late night activity
transactions.loc[
    transactions["hour"].between(1,4),
    "risk_score"
] += 2

# Rule 2 — Very high shopping purchases
transactions.loc[
    (transactions["merchant_category"]=="Shopping") &
    (transactions["amount"]>600),
    "risk_score"
] += 3

# Rule 3 — Online + Mobile Pay
transactions.loc[
    (transactions["channel"]=="Online") &
    (transactions["payment_method"]=="Mobile Pay"),
    "risk_score"
] += 2

# Rule 4 — Travel during night
transactions.loc[
    (transactions["merchant_category"]=="Travel") &
    (transactions["hour"]<5),
    "risk_score"
] += 2

transactions["risk_score"].value_counts().sort_index()

risk_score
0    784398
2    167653
3     19300
4     23545
5      4407
6       583
7       114
Name: count, dtype: int64

In [16]:
# Fraud Label


transactions["is_fraud"] = (
    transactions["risk_score"] >= 3
).astype(int)

transactions["is_fraud"].value_counts()

is_fraud
0    952051
1     47949
Name: count, dtype: int64

In [17]:
fraud_rate = (
    transactions["is_fraud"].mean()*100
)

print(f"Fraud Rate: {fraud_rate:.2f}%")

Fraud Rate: 4.79%


In [18]:
# Export Final Datasets

transactions.to_csv("/kaggle/working/bank_transactions.csv", index=False)
customers.to_csv("/kaggle/working/customers.csv", index=False)
merchants.to_csv("/kaggle/working/merchants.csv", index=False)

print("All datasets exported successfully!")

All datasets exported successfully!
